# How a model learns

Room310 · Deep learning foundations

## Goal

Train weight and bias yourself, so PyTorch's training loop will not feel like a magic spell.

## Setup

Run this notebook from top to bottom. It is self-contained and uses only synthetic teaching data. No credentials or dataset downloads are needed. Save a copy before editing.

The first two lessons need only standard Python.

## Steps

### 1. Give a mistake a number

A **loss function** measures how far predictions are from targets. For a numerical prediction, one useful loss is squared error: `(prediction - target) ** 2`. Squaring makes the error nonnegative and penalizes large misses more strongly.

For several examples, we average those squared errors. This is **mean squared error**, or MSE. A lower loss is better on the examples being measured; it does not automatically mean the model will work on new examples.

In [1]:
prediction = 4.0
target = 7.0
loss = (prediction - target) ** 2
print("Squared error:", loss)

Squared error: 9.0


**Check your result:** The loss is 9.0, not -3.0. Try prediction = 6.0 and then 8.0. Both are one unit away and have loss 1.0.

### 2. Find which direction is downhill

A **derivative** is a local slope: how much a result changes when one input changes a little. A **gradient** collects these slopes for the parameters. You do not need a calculus course to try a slope numerically: make a tiny nudge, measure the change in loss, and divide by the size of the nudge.

A positive slope says increasing the parameter raises the loss locally. To head downhill, move in the opposite direction. The **learning rate** controls the size of that move. Too large can overshoot; too small can take a long time.

In [2]:
def loss_at(weight):
    prediction = weight * 3.0 + 1.0
    return (prediction - 7.0) ** 2

weight = 0.0
epsilon = 0.0001
slope = (loss_at(weight + epsilon) - loss_at(weight - epsilon)) / (2 * epsilon)
new_weight = weight - 0.05 * slope
print("Slope:", round(slope, 3))
print("Loss before:", loss_at(weight))
print("Loss after:", round(loss_at(new_weight), 3))

Slope: -36.0
Loss before: 36.0
Loss after: 0.36


**Check your result:** The slope is approximately -36.0. The weight increases to about 1.8 and the loss falls from 36.0 to about 0.36.

### 3. Repeat the update: a training loop

Here the invented data follows `target = 2 * x + 1`. We know that rule, but the training calculation only receives input/target pairs.

For one example, let `error = weight * x + bias - target`. Its squared-error slope with respect to weight is `2 * error * x`; with respect to bias it is `2 * error`. Average each slope across the examples, then update both parameters. This is the **chain rule** in action: a change in weight changes the prediction, which changes the loss.

One **epoch** is one pass over the training examples. This tiny course uses the whole dataset as one batch, so one epoch is also one update. Larger datasets usually take several batch updates per epoch.

In [3]:
xs = [-2.0, -1.0, 0.0, 1.0, 2.0]
targets = [2 * x + 1 for x in xs]
weight, bias = 0.0, 0.0
learning_rate = 0.1

for epoch in range(60):
    errors = [weight * x + bias - y for x, y in zip(xs, targets)]
    loss = sum(error ** 2 for error in errors) / len(xs)
    grad_weight = sum(2 * error * x for error, x in zip(errors, xs)) / len(xs)
    grad_bias = sum(2 * error for error in errors) / len(xs)
    weight -= learning_rate * grad_weight
    bias -= learning_rate * grad_bias
    if epoch % 20 == 0:
        print(f"Epoch {epoch:2d} | loss before update: {loss:.6f}")

print(f"Learned weight: {weight:.3f}, bias: {bias:.3f}")
print(f"Prediction for x=3: {weight * 3 + bias:.3f}")

Epoch  0 | loss before update: 9.000000
Epoch 20 | loss before update: 0.000133
Epoch 40 | loss before update: 0.000000
Learned weight: 2.000, bias: 1.000
Prediction for x=3: 7.000


**Check your result:** Weight should approach 2.000, bias 1.000, and the prediction for 3 should approach 7.000. This result is for our simple invented line, not proof of real-world accuracy.

## Checks

Training repeats prediction, loss, gradients, and a small parameter update. PyTorch will calculate the gradients for us.

Compare your output with each check above. Explain unexpected results before moving on.

## Next Steps

### Practice & explain

### Learn a different line

Change the targets to -3 * x + 0.5. Train again from weight = bias = 0. Print the learned parameters and a prediction for x = 1. Your prediction should be close to -2.5.

<details><summary>Need a hint?</summary>

Change the data rule, not the gradient formulas. The formulas work for either line.

</details>

In [4]:
# Your experiment or explanation goes here.


### Keep an experiment log

Compare learning rates 0.01, 0.1, and 0.8 for the original data. Reset the parameters for each run. Record the final loss and explain which rate learns slowly and which is unstable.

<details><summary>Need a hint?</summary>

An increasing loss is a useful observation, not a reason to hide the result. Move in smaller steps if updates overshoot.

</details>

In [5]:
# Your experiment or explanation goes here.


### References

- [PyTorch · automatic differentiation](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
- [PyTorch · optimizing model parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)